# Week 5 - End-to-End Data Engineering Capstone: The Intern-Market Data Product

**This is the final week, and it is one thing: a complete data pipeline that covers most of what you
learned in Weeks 1-4 - built by you, from scratch, on live data.**

In Weeks 3-4 you built the medallion (bronze -> silver -> gold) and a star schema on tidy CBS data.
Here you do it all again on **live job-market data**, to answer a question you care about right now -
finding an internship:

> **"Which companies hire data and software interns, where in the Netherlands and Europe, in which
> kinds of roles - and how does the intern market compare to the job market as a whole?"**

**What is new this week:** you build *everything* (no tables are handed to you), the data is **live**
(real job posts, not a CSV), and it is big and growing - so you use **partitioning**, **columnar
files**, and **safe re-runnable loads** for real.

Every idea keeps the Week-4 rhythm - **a problem -> a worked solution -> your turn** - so watch for the
**Challenge** boxes, and for the *peek at the data* at each stage.

## The big picture - the whole pipeline in one diagram

You already know every tool; this week just connects them. **Six messy live sources become one clean
star you can chart.** Each block below is a Lab - find a cell on this map first when it feels complex.

**Week 1 in one breath:** the job sites we pull from are built to *run* a business (**OLTP** - many tiny
reads and writes, handled by various **data roles**); what we build is for *analysing* it (**OLAP** -
big read-only queries). The pipeline is the classic **three tiers** (source -> storage -> serving),
saved as files, and our file writes copy a simple version of a database's **ACID** safety. We store data
**by column** (Parquet), not by row, which is why the charts at the end are fast.

<div style="font-family:'Segoe UI',system-ui,sans-serif;color:#334155;margin:10px 0;"><div style="font-size:15px;font-weight:700;margin-bottom:12px;">Data pipeline <span style="font-weight:400;color:#94A3B8;font-size:12.5px;">&nbsp;each step reads the box on its left and writes the box on its right</span></div><div style="overflow-x:auto;padding-bottom:8px;"><table style="border-collapse:collapse;border:none;background:transparent;"><tbody><tr><td style="border:none;padding:0;vertical-align:top;"><div style="border:1.5px solid #94A3B8;border-radius:10px;background:#fff;width:152px;padding:9px 10px;vertical-align:top;"><div style="font-size:9px;letter-spacing:1px;color:#94A3B8;font-weight:700;text-align:center;">SOURCES &middot; the live web</div><div style="display:flex;justify-content:space-between;border:1px solid #CBD5E1;border-radius:6px;background:#F8FAFC;padding:3px 7px;font-size:10px;margin-top:4px;"><span>The Muse</span><span style="color:#94A3B8;">API</span></div><div style="display:flex;justify-content:space-between;border:1px solid #CBD5E1;border-radius:6px;background:#F8FAFC;padding:3px 7px;font-size:10px;margin-top:4px;"><span>Greenhouse</span><span style="color:#94A3B8;">ATS</span></div><div style="display:flex;justify-content:space-between;border:1px solid #CBD5E1;border-radius:6px;background:#F8FAFC;padding:3px 7px;font-size:10px;margin-top:4px;"><span>Lever</span><span style="color:#94A3B8;">ATS</span></div><div style="display:flex;justify-content:space-between;border:1px solid #CBD5E1;border-radius:6px;background:#F8FAFC;padding:3px 7px;font-size:10px;margin-top:4px;"><span>Ashby</span><span style="color:#94A3B8;">ATS</span></div><div style="display:flex;justify-content:space-between;border:1px solid #CBD5E1;border-radius:6px;background:#F8FAFC;padding:3px 7px;font-size:10px;margin-top:4px;"><span>Recruitee</span><span style="color:#94A3B8;">ATS</span></div><div style="display:flex;justify-content:space-between;border:1px solid #CBD5E1;border-radius:6px;background:#F8FAFC;padding:3px 7px;font-size:10px;margin-top:4px;"><span>Arbeitnow</span><span style="color:#94A3B8;">board</span></div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border-radius:10px;background:#156082;color:#fff;width:120px;padding:11px 9px;text-align:center;"><div style="font-size:8.5px;letter-spacing:1.2px;opacity:.8;">PIPELINE STEP</div><div style="font-weight:700;font-size:13px;margin-top:4px;">Extract</div><div style="font-size:9.5px;opacity:.9;margin-top:6px;line-height:1.45;">pull each feed,<br>save it raw</div><div style="font-size:9px;opacity:.7;margin-top:7px;">Lab 0</div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border:1px solid #E2E8F0;border-top:4px solid #156082;border-radius:10px;background:#fff;width:170px;padding:9px 11px;vertical-align:top;"><div style="font-weight:700;color:#156082;font-size:12.5px;">Bronze</div><div style="font-size:9px;color:#94A3B8;margin-bottom:3px;">raw files, untouched</div><div style="font-size:10px;color:#475569;"><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">muse_*.json</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">ats_greenhouse_*.json</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">ats_lever / ashby /</span><br>&nbsp;&nbsp;<span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">recruitee_*.json</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">arbeitnow.json</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">_manifest.json</span> (SHA-256)</div></div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border-radius:10px;background:#156082;color:#fff;width:120px;padding:11px 9px;text-align:center;"><div style="font-size:8.5px;letter-spacing:1.2px;opacity:.8;">PIPELINE STEP</div><div style="font-weight:700;font-size:13px;margin-top:4px;">Clean</div><div style="font-size:9.5px;opacity:.9;margin-top:6px;line-height:1.45;">map &rarr; one model,<br>type, quarantine,<br>classify, partition</div><div style="font-size:9px;opacity:.7;margin-top:7px;">Lab 1</div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border:1px solid #E2E8F0;border-top:4px solid #156082;border-radius:10px;background:#fff;width:168px;padding:9px 11px;vertical-align:top;"><div style="font-weight:700;color:#156082;font-size:12.5px;">Silver</div><div style="font-size:9px;color:#94A3B8;margin-bottom:3px;">one clean table</div><div style="font-size:10px;color:#475569;"><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">postings/</span> (Parquet, by month)</div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">bridge_posting_category</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">_quarantine/</span> (bad rows)</div></div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border-radius:10px;background:#156082;color:#fff;width:120px;padding:11px 9px;text-align:center;"><div style="font-size:8.5px;letter-spacing:1.2px;opacity:.8;">PIPELINE STEP</div><div style="font-weight:700;font-size:13px;margin-top:4px;">Model</div><div style="font-size:9.5px;opacity:.9;margin-top:6px;line-height:1.45;">grain, dims,<br>facts, keys</div><div style="font-size:9px;opacity:.7;margin-top:7px;">Lab 2</div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border:1px solid #E2E8F0;border-top:4px solid #156082;border-radius:10px;background:#fff;width:178px;padding:9px 11px;vertical-align:top;"><div style="font-weight:700;color:#156082;font-size:12.5px;">Gold</div><div style="font-size:9px;color:#94A3B8;margin-bottom:3px;">the star schema</div><div style="font-size:10px;color:#475569;"><div style="margin-top:4px;line-height:1.35;">&bull; 6 dims: <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">company, location,</span></div><div style="margin-top:4px;line-height:1.35;">&bull; &nbsp;&nbsp;<span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">level, category,</span></div><div style="margin-top:4px;line-height:1.35;">&bull; &nbsp;&nbsp;<span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">discipline, date</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">fact_posting</span></div><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">fact_company_month</span></div></div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border-radius:10px;background:#156082;color:#fff;width:120px;padding:11px 9px;text-align:center;"><div style="font-size:8.5px;letter-spacing:1.2px;opacity:.8;">PIPELINE STEP</div><div style="font-weight:700;font-size:13px;margin-top:4px;">Serve</div><div style="font-size:9.5px;opacity:.9;margin-top:6px;line-height:1.45;">semantic view<br>+ charts</div><div style="font-size:9px;opacity:.7;margin-top:7px;">Lab 4</div></div></td><td style="border:none;padding:0 5px;vertical-align:middle;color:#94A3B8;font-size:19px;">&#8594;</td><td style="border:none;padding:0;vertical-align:top;"><div style="border:1px solid #E2E8F0;border-top:4px solid #156082;border-radius:10px;background:#fff;width:158px;padding:9px 11px;vertical-align:top;"><div style="font-weight:700;color:#156082;font-size:12.5px;">The answer</div><div style="font-size:9px;color:#94A3B8;margin-bottom:3px;">what we deliver</div><div style="font-size:10px;color:#475569;"><div style="margin-top:4px;line-height:1.35;">&bull; <span style="font-family:Consolas,monospace;font-size:9.5px;color:#0f3a52;">vw_intern_market</span> (view)</div><div style="margin-top:4px;line-height:1.35;">&bull; charts: the market,</div><div style="margin-top:4px;line-height:1.35;">&bull; &nbsp;&nbsp;data/AI/ML, NL + Europe</div></div></div></td></tr></tbody></table></div><div style="margin-top:8px;padding:8px 12px;background:#F1F5F9;border-left:3px solid #5EAACD;border-radius:6px;font-size:11.5px;color:#475569;display:inline-block;"><b>Lab 3 &middot; Scalability</b> &mdash; keeps the Silver &amp; Gold steps fast as the data grows (partitioned Parquet + safe, re-runnable loads).</div></div>

## Part 0 - Just enough about scale

Before we build, one question shapes everything this week: **what happens when the data gets big?**

| Approach | What it means | Example |
|---|---|---|
| **Scale up** | use a bigger machine | more CPU and memory on one server |
| **Scale out** | use many machines | a cluster of servers (Spark, Databricks) |

A simple rule of thumb for analytics work:

```
Small   (under ~10 GB):  one machine - DuckDB or pandas (what we use here)
Medium  (10-100 GB):     one big machine + Parquet split into folders
Large   (over ~1 TB):    many machines - a cluster (Spark / Databricks / cloud warehouse)
```

One machine with DuckDB and Parquet gets you surprisingly far. Three tricks make that possible, and
you use all three in this notebook:

1. **Columnar files (Parquet)** - a query reads only the columns it needs, not the whole row.
2. **Partitioning** - split a table into folders (say, one per month). A query filtered by month opens
   only that folder and skips the rest.
3. **Safe, repeatable loads** - add today's new data without redoing all the old data, and running it
   twice never creates duplicates.

> **Challenge: Why?** Your manager says *"just put everything in one giant CSV, it's simpler."* Give
> two clear reasons that hurts you at 50 GB.
> <details><summary>Answer</summary>A CSV stores whole rows and is not compressed, so every query reads
> the *entire* file - even to look at one column - and it cannot skip data it does not need. Parquet
> plus partitioning reads only the columns and only the folders a query needs, often just 1-10% of the file.</details>

> **What this notebook does *not* cover - and why.** To keep it to one machine you can run yourself, we
> leave out things you would meet in a bigger setup: a full **Delta Lake / lakehouse** (we use Parquet
> plus a folder rewrite as a basic stand-in - you built a real Delta table by hand in **Week 2**),
> **streaming / real-time CDC**, pipeline **orchestration** (Airflow, Dagster), and **cloud warehouses
> / clusters** (Spark, Databricks, Snowflake). The ideas here carry over to all of those - only the
> tools change.

## Setup - medallion folders

Same three layers as Week 3/4, now for our own project under `week5/data/`.


In [ ]:
# What: set up our tools and the three medallion folders (bronze / silver / gold).
# Why:  one place for the libraries and folders, so every later cell can just read and write there.

import json, time, shutil, re          # json: read/write files | time: polite delays | shutil: delete folders | re: text cleanup
from pathlib import Path               # Path: handle file paths the same way on Windows / Mac / Linux

import duckdb                          # our SQL engine - queries Parquet/JSON in memory, no server needed
import pandas as pd                    # tables (DataFrames) for the Python side of the work
import matplotlib.pyplot as plt        # draws the charts at the end

try:
    import requests                    # the nicer HTTP library - use it if it is installed
    _HTTP = "requests"                 # remember that we picked requests
except Exception:
    import urllib.request              # fallback that ships with Python, so the notebook always runs
    _HTTP = "urllib"                   # remember that we fell back to urllib

# make every chart share one clean style (size, faint grid, no top/right border, readable font)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
TEAL, SLATE = "#156082", "#334155"     # our two colours, reused in every chart

DATA = Path("week5/data")              # root folder for this project's data
BRONZE, SILVER, GOLD = DATA / "bronze", DATA / "silver", DATA / "gold"   # the three medallion layers
for d in (BRONZE, SILVER, GOLD):
    d.mkdir(parents=True, exist_ok=True)   # create each folder if missing (safe to re-run)

con = duckdb.connect()                 # open ONE in-memory DuckDB connection we reuse all notebook
print("Medallion folders ready under", DATA.as_posix())

## Lab 0 - Bronze: collect the raw job data

**The problem.** Our data lives on the live web, not in a neat file. So first we save an exact,
untouched copy of whatever each website sends back. That copy is the **bronze layer**. Keeping it lets
us re-run and check our work later, without hitting the websites again.

**What we collect (all free, no login needed):**
- **The Muse** - internships and a general sample of jobs, plus a sweep of European cities (Amsterdam,
  Berlin, London, Paris, and more) so we have enough Dutch and European roles. It gives us company,
  location, role type, level, and date.
- **Company career feeds** - public, no-login job feeds from real **Dutch and European employers**
  (Adyen, bol.com, Databricks, Snowflake, CZ, Hot ITem, FreshMinds, UMC Utrecht, and more) through
  four different systems: **Greenhouse, Lever, Ashby and Recruitee** (Recruitee is a Dutch one). Each
  formats its data differently - on purpose, so you learn to handle that. You can edit these lists.
- **Arbeitnow** - a free European job board.

We collect **broadly** on purpose: grab lots of jobs now, and only narrow down to internships, the
Netherlands, or data roles *later*. Filtering too early throws away data we might want.

We also make it **safe to re-run**: if a file is already saved, we skip downloading it again.

**Step 1 - small download helpers.** A tiny function to fetch a URL, and one that walks through The Muse page by page. We reuse both below.

In [ ]:
# What: two small download helpers: fetch any URL as JSON, and walk The Muse page by page.
# Why:  we reuse these for every source, so the download logic is written once, not repeated.

UA = {"User-Agent": "InHolland-DataEng-Course/1.0 (educational)"}   # polite ID we send with each request

def _get(url):                         # fetch one URL and hand back parsed JSON
    if _HTTP == "requests":            # if the requests library is available...
        return requests.get(url, headers=UA, timeout=40).json()   # ...GET it and parse JSON (give up after 40s)
    req = urllib.request.Request(url, headers=UA)                  # otherwise build the request by hand
    return json.loads(urllib.request.urlopen(req, timeout=40).read())   # open, read the bytes, parse JSON

def fetch_muse(params, pages):         # The Muse returns results in pages; collect up to `pages` of them
    rows = []                          # every job goes in here
    for pg in range(1, pages + 1):     # page 1, 2, 3, ... up to the limit
        d = _get(f"https://www.themuse.com/api/public/jobs?{params}&page={pg}")   # ask for this page
        rows += d.get("results", [])   # add this page's jobs to our list
        if pg >= d.get("page_count", 1):   # reached the last real page? stop early
            break
        time.sleep(0.25)               # short pause between requests, to be polite to the API
    return rows                        # give back everything we collected

**Step 2 - pull The Muse.** Internships + a general sample, then a pass over European cities so the NL/EU slice has real volume. Idempotent: if a file already exists we skip the download.

In [ ]:
# What: download The Muse jobs (internships, a general sample, and European cities) into bronze.
# Why:  keep an exact raw copy so we can re-run without re-hitting the API; pull broadly, filter later.

# The Muse: internships + a general sample
if not (BRONZE / "muse_internships.json").exists():        # only download if we don't already have the file
    print("fetching The Muse internships ...")
    (BRONZE / "muse_internships.json").write_text(json.dumps(fetch_muse("level=Internship", 25), ensure_ascii=False), encoding="utf-8")   # 25 pages of internships -> save raw JSON
if not (BRONZE / "muse_general.json").exists():            # same guard for the general sample
    print("fetching The Muse general ...")
    (BRONZE / "muse_general.json").write_text(json.dumps(fetch_muse("", 25), ensure_ascii=False), encoding="utf-8")   # 25 pages of ALL jobs = our 'wider market' denominator

# The Muse, focused on European cities (a BROAD pull - we filter to NL/EU later, never at the fetch)
from urllib.parse import quote          # quote() makes a city name safe to drop inside a URL
EU_CITIES = ["Amsterdam, Netherlands", "Rotterdam, Netherlands", "Berlin, Germany",
             "London, United Kingdom", "Dublin, Ireland", "Paris, France", "Barcelona, Spain"]   # cities to sweep
for _city in EU_CITIES:                 # one bronze file per city
    dst = BRONZE / ("muse_eu_" + re.sub(r"[^a-z]+", "_", _city.lower()).strip("_") + ".json")   # safe filename built from the city name
    if not dst.exists():                # skip if already downloaded
        print(f"fetching The Muse {_city} ...")
        dst.write_text(json.dumps(fetch_muse("location=" + quote(_city), 12), ensure_ascii=False), encoding="utf-8")   # 12 pages of jobs in this city

# ...and an internship-focused EU pull, so the NL/EU slice has real volume (still broad: we keep all)
for _city in EU_CITIES + ["Munich, Germany"]:   # same cities (+ Munich), internships only this time
    dst = BRONZE / ("muse_euint_" + re.sub(r"[^a-z]+", "_", _city.lower()).strip("_") + ".json")   # separate filename so it doesn't clash
    if not dst.exists():
        print(f"fetching The Muse internships in {_city} ...")
        dst.write_text(json.dumps(fetch_muse("location=" + quote(_city) + "&level=Internship", 15), ensure_ascii=False), encoding="utf-8")   # 15 pages of internships in this city

**Step 3 - pull the ATS feeds + Arbeitnow.** Greenhouse, Lever, Ashby and Recruitee (each a *different* JSON shape) plus the Arbeitnow EU board. Each is wrapped in try/except so one dead feed can't stop the pull.

**Step 3a - Greenhouse + Arbeitnow.** A few target companies' Greenhouse boards, plus the free Arbeitnow EU feed (paginated).

In [ ]:
# What: download the Greenhouse career feeds for our Dutch / EU companies.
# Why:  Greenhouse hands out jobs as open JSON (no login); try/except = one dead feed can't stop the rest.

# Greenhouse career feeds - add or remove company names here
ATS_COMPANIES = ["adyen", "bolcom", "databricks", "mongodb", "dept", "lucidmotors",
                 "suitsupply", "artefact", "edgeconnex", "wikimedia", "medecinssansfrontieres"]   # Greenhouse 'board' names
for slug in ATS_COMPANIES:              # one company at a time
    dst = BRONZE / f"ats_greenhouse_{slug}.json"   # where this company's jobs will be saved
    if dst.exists():                    # already downloaded? skip it (idempotent)
        continue
    try:
        print(f"fetching ATS {slug} ...")
        dst.write_text(json.dumps(_get(f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs?content=true"), ensure_ascii=False), encoding="utf-8")   # Greenhouse's public jobs endpoint
        time.sleep(0.3)                 # small pause between companies
    except Exception as e:              # a wrong slug or a down feed lands here...
        print(f"  {slug} skipped ({e})")   # ...note it and carry on, no crash

# Arbeitnow - a free European job-board API (one endpoint, paginated)
if not (BRONZE / "arbeitnow.json").exists():   # skip if we already have it
    print("fetching Arbeitnow (EU) ...")
    eu = []                             # collect all pages here
    for pg in range(1, 6):              # pages 1..5
        try:
            eu += _get(f"https://www.arbeitnow.com/api/job-board-api?page={pg}").get("data", [])   # add this page's jobs
        except Exception:
            break                       # a page failed -> stop paging
        time.sleep(0.3)
    (BRONZE / "arbeitnow.json").write_text(json.dumps(eu, ensure_ascii=False), encoding="utf-8")   # save the combined result

**Step 3b - Lever, Ashby & Recruitee.** Three more ATS providers - each a *different* JSON shape, normalised later in silver.

In [ ]:
# What: download the Lever, Ashby and Recruitee feeds (each a different JSON shape).
# Why:  more open feeds = more real Dutch employers; different shapes are on purpose, so we practise mapping them.

# More open ATS feeds - each provider a DIFFERENT JSON shape, on purpose.
LEVER_COMPANIES     = ["mistral", "spotify", "bloomon", "immuta", "hso"]   # companies on Lever's global host
LEVER_EU_COMPANIES  = ["prima", "prosus"]                                   # companies on Lever's EU host
ASHBY_COMPANIES     = ["snowflake", "pliant", "everfield", "reson8"]        # companies on Ashby
RECRUITEE_COMPANIES = ["alliade", "cz", "hiltermannlease", "freshminds", "hotitem", "polaroid",
                       "umcutrecht", "visma", "revodata", "teamvalue", "tataconsultancyservices", "kekkilabvb"]   # Dutch companies on Recruitee

def _save_lever(slug, host):            # Lever has two hosts (global + EU); one helper covers both
    dst = BRONZE / f"ats_lever_{slug}.json"
    if dst.exists():                    # already saved? skip
        return
    try:
        dst.write_text(json.dumps(_get(f"https://{host}/v0/postings/{slug}?mode=json"), ensure_ascii=False), encoding="utf-8")   # Lever's public postings endpoint
        time.sleep(0.3)
    except Exception as e:
        print(f"  lever/{slug} skipped ({e})")   # bad slug/host -> note and continue

for slug in LEVER_COMPANIES:
    _save_lever(slug, "api.lever.co")           # global host
for slug in LEVER_EU_COMPANIES:
    _save_lever(slug, "api.eu.lever.co")        # EU host (some EU companies live here)

for slug in ASHBY_COMPANIES:            # Ashby = different URL + different JSON shape
    dst = BRONZE / f"ats_ashby_{slug}.json"
    if not dst.exists():
        try:
            dst.write_text(json.dumps(_get(f"https://api.ashbyhq.com/posting-api/job-board/{slug}"), ensure_ascii=False), encoding="utf-8")   # Ashby's public board endpoint
            time.sleep(0.3)
        except Exception as e:
            print(f"  ashby/{slug} skipped ({e})")

# Recruitee - a Dutch ATS used by many NL employers (yet another JSON shape)
for slug in RECRUITEE_COMPANIES:
    dst = BRONZE / f"ats_recruitee_{slug}.json"
    if not dst.exists():
        try:
            dst.write_text(json.dumps(_get(f"https://{slug}.recruitee.com/api/offers/"), ensure_ascii=False), encoding="utf-8")   # each company has its own <slug>.recruitee.com
            time.sleep(0.3)
        except Exception as e:
            print(f"  recruitee/{slug} skipped ({e})")

**Step 4 - list what we saved, and fingerprint it.** Show the files, then save a short SHA-256
fingerprint of each one in `_manifest.json`.

*What is a fingerprint?* A **SHA-256 hash** is a short, fixed-length string calculated from a file's
exact bytes. The same bytes always produce the same string, and changing even one character produces a
completely different one. So storing the fingerprint (not the whole file) lets us later **prove a bronze
file has not changed** and record where each file came from - the integrity and lineage idea from Week 3.

In [ ]:
# What: list what landed, then save a SHA-256 fingerprint of each bronze file in a manifest.
# Why:  the fingerprint proves a file has not changed and records where it came from (Week-3 lineage / audit).

print("\nBronze landed:")
for p in sorted(BRONZE.glob("*.json")):     # every JSON file now in bronze
    if p.name.startswith("_"):              # skip our own helper files (like _manifest.json)
        continue
    print(f"  {p.name:32} {p.stat().st_size // 1024:>5} KB")   # show each file's name and size in KB

# Extraction metadata + integrity (Week 3): SHA-256 every bronze file, write a run manifest (lineage)
import hashlib                              # provides the SHA-256 hashing function
from datetime import datetime, timezone     # to stamp this run with a UTC time
run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")   # a unique id for THIS download run
manifest = {"run_id": run_id, "files": {    # the manifest = run id + one entry per file
    p.name: {"bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16]}   # size + short fingerprint of the file's bytes
    for p in sorted(BRONZE.glob("*.json")) if not p.name.startswith("_")}}   # for every real bronze file
(BRONZE / "_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")   # save the manifest beside the data
print(f"\nrun_id = {run_id}  |  SHA-256 lineage written for {len(manifest['files'])} bronze files")

### What does the raw data actually look like?

Bronze holds the data exactly as each site sent it - and every source uses a **different JSON shape**.
Before we clean anything, let's *look* at it (the "know your data" habit from Week 3). The cell below
prints one raw posting from two different feeds, so you can see the mess we map onto one model in Lab 1.

In [ ]:
# What: peek at one raw posting from two different sources, straight from bronze.
# Why:  "know your data" (Week 3) - see the different JSON shapes BEFORE we map them to one model.

import itertools

def _first_record(path, key=None):         # pull the first job object out of a bronze file
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if key:                                # some feeds nest the list under a key (e.g. "jobs" / "offers")
        data = data.get(key, [])
    return data[0] if isinstance(data, list) and data else data

def _preview(d, n=12):                      # show only the first n top-level fields, trimmed, so output stays short
    if not isinstance(d, dict):
        return d
    out = {}
    for k, v in itertools.islice(d.items(), n):
        s = json.dumps(v, ensure_ascii=False)
        out[k] = (s[:90] + " ...") if len(s) > 90 else v
    return out

muse_file = sorted(BRONZE.glob("muse_*.json"))[0]            # any Muse file
gh_file   = sorted(BRONZE.glob("ats_greenhouse_*.json"))[0]  # any Greenhouse file
print("THE MUSE - one raw record (first fields):")
print(json.dumps(_preview(_first_record(muse_file)), indent=2, ensure_ascii=False))
print("\nGREENHOUSE - one raw record (first fields):")
print(json.dumps(_preview(_first_record(gh_file, key="jobs")), indent=2, ensure_ascii=False))
print("\nNotice: different field names, different nesting. Lab 1 maps all six sources onto ONE shape.")

> **Looking back at Weeks 2 and 3 (storage and ingestion).**
> - **Data lake, swamp, lakehouse (Week 2):** the bronze folder is a *data lake landing zone* - the raw drop-off point for data. It stays a useful lake instead of turning into a "swamp" only because we keep it raw and never edit it by hand.
> - **The four ways to ingest data (Week 3):** full load, incremental, change-data-capture, and paginated. Here we read The Muse and Arbeitnow page by page, and pull each company feed in full.
> - **Control flow vs data flow (Week 3):** control flow is the *order* we run the steps in (and the "skip if the file already exists" checks). Data flow is what happens to each record *inside* a step.
> - **File formats (Week 2):** bronze is raw **JSON**, exactly as each source sent it. Silver will switch to **Parquet**, which is faster to query - the next lab shows why.
> - **Knowing where data came from (Week 3):** we save a **SHA-256** fingerprint of every bronze file in `_manifest.json`. That lets you prove a file has not changed and trace where it came from.

> **Challenge: Why?** We collect internships *and* a general sample of all jobs. Why not just collect internships?
> <details><summary>Answer</summary>Our main number is **internship share**: internships divided by all
> jobs. Without the "all jobs" part you can count internships, but you cannot tell whether a company or
> city has *unusually many*. Rule of thumb: collect the data your numbers need, not just the rows that look interesting.</details>

## Lab 1 - Silver: put every source into one shared shape

**The problem.** Bronze holds **six different shapes**. The Muse, Greenhouse, Lever, Ashby,
Arbeitnow and Recruitee each use different field names, nest their data differently, and even write dates differently
(some use text dates, some use numbers). You cannot build a model on top of that mess.

**The fix: one shared shape (a common model).** Decide on *one* layout for a job posting, then write a
short piece of code for each source that copies its fields into that layout. Every source ends up in the
**same** silver table, with a `source` column saying where each row came from. This is the core of
combining data: do the messy matching **once**, here. Adding another source later is just one more small
mapping - nothing further down the pipeline has to change.

| Our shared shape | The Muse | Greenhouse | Lever | Ashby | Arbeitnow | Recruitee |
|---|---|---|---|---|---|---|
| `title` | `name` | `title` | `text` | `title` | `title` | `title` |
| `company` | `company.name` | `company_name` | *(board)* | *(board)* | `company_name` | `company_name` |
| `job_family` | `categories[0]` | `departments[0]` | `categories.team` | `department` | `tags[0]` | `department` |
| `location` | `locations[0]` | `location.name` | `categories.location` | `location` | `location` | `location` |
| `posted_date` | `publication_date` | `first_published` | `createdAt` (ms) | `publishedAt` | `created_at` (epoch s) | `published_at` |
| `level` | `levels[0]` | *guess from title* | *guess* | *guess* | *guess* | *guess* |

We also keep the "a posting can have several job families" link in a separate **bridge** table, and we
save silver as **Parquet split into one folder per month** - our first scalability move.

> **Challenge: Why?** Why use one shared shape instead of just keeping six separate tables?
> <details><summary>Answer</summary>With six tables, every later query and chart has to understand all
> six shapes - and breaks whenever a source changes. The shared shape pushes all that mess into one
> small mapping per source. Everything after it (the model, the charts, the report) is written just once.
> It is the same idea as splitting facts and dimensions in Week 4: do the organising work one time.</details>

**Step 1 - set up the shared shape.** Define the intern keywords, a small helper that splits a location into city and country, and the empty lists we will fill from every source.

In [ ]:
# What: set up the shared 'common model': intern keywords, a location splitter, and empty lists to fill.
# Why:  we pour every source into the SAME shape; deciding that shape up front is the heart of combining data.

# words in a job title that signal an internship (English + Dutch: 'stage', 'werkstudent')
INT_HINTS = ("intern", "stage", "graduate", "working student", "university", "trainee", "werkstudent")

def split_location(s):                 # turn a free-text location into (city, country)
    parts = [p.strip() for p in str(s).split(",") if p.strip()]   # split on commas, drop blanks
    if len(parts) >= 2:                # e.g. "Amsterdam, Netherlands"
        return parts[0], parts[-1]     # first part = city, last part = country/region
    return (str(s).strip() or "Unknown", "Unknown")   # only one part -> city only, country Unknown

rows, cat_links = [], []               # rows = postings | cat_links = (posting, job-family) pairs for the bridge

**Step 2 - map The Muse & Greenhouse.** Read each source's fields onto the *one* common schema.

In [ ]:
# What: map The Muse and Greenhouse fields onto our common model.
# Why:  each source names things differently, so we translate once here; everything downstream then sees one shape.

# --- The Muse ---
for fn in sorted(BRONZE.glob("muse_*.json")):      # every Muse file we downloaded
    targeted = ("internship" in fn.name) or ("euint" in fn.name)   # was THIS file an internship-only pull? (biases share)
    for j in json.loads(fn.read_text(encoding="utf-8")):   # each job object in that file
        pid = "muse_" + str(j["id"])               # unique id, prefixed so it can't clash with other sources
        company = (j.get("company") or {}).get("name") or "Unknown"   # company name (nested), else Unknown
        levels = [l["name"] for l in j.get("levels", [])]     # seniority levels list
        cats = [c["name"] for c in j.get("categories", [])]   # categories list
        locs = [l["name"] for l in j.get("locations", [])]    # locations list
        rows.append(dict(posting_id=pid, source="themuse", company=company, title=j.get("name") or "",   # -> common columns
                         level=(levels[0] if levels else "Unknown"),     # first level, else Unknown
                         category=(cats[0] if cats else "Other"),        # first category, else Other
                         location=(locs[0] if locs else "Unknown"),      # first location, else Unknown
                         posted=j.get("publication_date"),               # when posted (ISO text)
                         url=(j.get("refs") or {}).get("landing_page"),    # apply link
                         from_intern_pull=targeted))  # True only for internship-targeted Muse files (fair-share filter)
        for c in cats:                 # a posting can list several categories...
            cat_links.append((pid, c)) # ...keep each as a bridge row

# --- Greenhouse ATS ---
for fn in BRONZE.glob("ats_greenhouse_*.json"):    # every Greenhouse file
    data = json.loads(fn.read_text(encoding="utf-8"))
    for j in data.get("jobs", []):                 # Greenhouse nests jobs under "jobs"
        pid = "gh_" + str(j["id"])                 # unique id with a 'gh_' prefix
        deps = [d.get("name") for d in j.get("departments", []) if d.get("name")]   # departments = our job family
        rows.append(dict(posting_id=pid, source="greenhouse",
                         company=j.get("company_name") or fn.stem.replace("ats_greenhouse_", "").title(),   # name, else from filename
                         title=j.get("title") or "", level="Unknown",    # no seniority field -> infer later
                         category=(deps[0] if deps else "Other"),        # first department, else Other
                         location=(j.get("location") or {}).get("name", "Unknown"),   # location is nested
                         posted=j.get("first_published") or j.get("updated_at"),       # first published, else updated
                         url=j.get("absolute_url"),                        # link to the posting
                         from_intern_pull=False))                          # Greenhouse = a full pull (interns + non-interns)
        for c in deps:
            cat_links.append((pid, c)) # one bridge row per department

**Step 3 - map Arbeitnow, Lever, Ashby & Recruitee.** Same target schema, different shapes - note the epoch-date helper for the millisecond/second timestamps.

In [ ]:
# What: map Arbeitnow, Lever, Ashby and Recruitee onto the same common model.
# Why:  same target columns, different source shapes - note the epoch-date helper for the numeric timestamps.

from datetime import datetime, timezone
def _epoch_iso(v, ms=False):           # turn a numeric unix timestamp into ISO text
    if v in (None, ""):                # nothing to convert
        return None
    return datetime.fromtimestamp(int(v) / (1000 if ms else 1), tz=timezone.utc).isoformat()   # ms? divide by 1000

# --- Arbeitnow (EU) -> common model ---
for j in json.loads((BRONZE / "arbeitnow.json").read_text(encoding="utf-8")):   # one file = a list of jobs
    pid = "arbeitnow_" + str(j.get("slug"))        # Arbeitnow uses a text 'slug' as id
    tags = j.get("tags") or []                     # tags act as our job family
    fam = tags[0] if tags else (j.get("job_types") or ["Other"])[0]   # first tag, else first job_type, else Other
    rows.append(dict(posting_id=pid, source="arbeitnow", company=j.get("company_name") or "Unknown",
                     title=j.get("title") or "", level="Unknown", category=fam,
                     location=j.get("location") or ("Remote" if j.get("remote") else "Unknown"),   # remote flag -> "Remote"
                     posted=_epoch_iso(j.get("created_at")), url=j.get("url"),   # created_at is unix SECONDS
                     from_intern_pull=False))                                     # full pull (not internship-targeted)
    for c in tags:
        cat_links.append((pid, c))

# --- Lever ATS -> common model (dates are epoch milliseconds) ---
for fn in BRONZE.glob("ats_lever_*.json"):         # every Lever file (global + EU share this prefix)
    comp = fn.stem.replace("ats_lever_", "").title()   # Lever gives no company name -> use the filename
    for j in json.loads(fn.read_text(encoding="utf-8")):   # a Lever file is a plain list of postings
        pid = "lever_" + str(j.get("id"))
        cat = j.get("categories") or {}            # team/department/location live under 'categories'
        fam = cat.get("team") or cat.get("department") or "Other"   # job family = team, else department
        rows.append(dict(posting_id=pid, source="lever", company=comp, title=j.get("text") or "",   # Lever's title field is 'text'
                         level="Unknown", category=fam, location=cat.get("location") or "Unknown",
                         posted=_epoch_iso(j.get("createdAt"), ms=True), url=j.get("hostedUrl"),   # createdAt is MILLISECONDS
                         from_intern_pull=False))                                  # full pull (not internship-targeted)
        cat_links.append((pid, fam))

# --- Ashby ATS -> common model ---
for fn in BRONZE.glob("ats_ashby_*.json"):
    comp = fn.stem.replace("ats_ashby_", "").title()   # company from filename
    for j in json.loads(fn.read_text(encoding="utf-8")).get("jobs", []):   # Ashby nests under 'jobs'
        pid = "ashby_" + str(j.get("id"))
        fam = j.get("department") or j.get("team") or "Other"   # department, else team
        rows.append(dict(posting_id=pid, source="ashby", company=comp, title=j.get("title") or "",
                         level="Unknown", category=fam, location=j.get("location") or "Unknown",
                         posted=j.get("publishedAt"), url=j.get("jobUrl") or j.get("applyUrl"),   # Ashby date is already ISO
                         from_intern_pull=False))                                  # full pull (not internship-targeted)
        cat_links.append((pid, fam))

# --- Recruitee ATS (Dutch) -> common model ---
for fn in BRONZE.glob("ats_recruitee_*.json"):
    for j in json.loads(fn.read_text(encoding="utf-8")).get("offers", []):   # Recruitee nests under 'offers'
        pid = "recruitee_" + str(j.get("id"))
        dt = (j.get("published_at") or j.get("created_at") or "").replace(" UTC", "").replace(" ", "T")   # "... UTC" -> ISO "...T..."
        fam = j.get("department") or "Other"
        rows.append(dict(posting_id=pid, source="recruitee", company=j.get("company_name") or "Unknown",
                         title=j.get("title") or "", level="Unknown", category=fam,
                         location=j.get("location") or j.get("city") or ("Remote" if j.get("remote") else "Unknown"),
                         posted=(dt or None), url=j.get("careers_url"),
                         from_intern_pull=False))                                  # full pull (not internship-targeted)
        cat_links.append((pid, fam))

**Step 4 - assemble, clean, type & quarantine.** Build the DataFrame, standardise types, and route rows with an unparseable date to the dead-letter folder instead of dropping them silently.

In [ ]:
# What: build one DataFrame, standardise the types and dates, and quarantine rows with a bad date.
# Why:  clean once, in silver; rows that fail go to a dead-letter folder instead of disappearing silently (data quality).

df = pd.DataFrame(rows)                 # turn the list of dicts into one table

# clean + type
df["company"] = df["company"].fillna("Unknown").astype(str).str.strip()   # no blanks, as text, trimmed
df["title"] = df["title"].fillna("").astype(str).str.strip()              # title as trimmed text
df["posted_date"] = pd.to_datetime(df["posted"], errors="coerce", utc=True, format="ISO8601").dt.tz_localize(None)   # parse dates; bad ones become NaT

# QUARANTINE (Week 3): route rows that fail validation to a dead-letter folder - never silently drop
_bad = df[df["posted_date"].isna()].copy()        # rows whose date could not be parsed
_bad["_reason"] = "unparseable posted_date"        # record WHY each was quarantined
(SILVER / "_quarantine").mkdir(exist_ok=True)      # make the dead-letter folder if needed
_bad.to_parquet(SILVER / "_quarantine" / "bad_postings.parquet", index=False)   # save the bad rows for review
print(f"quarantined {len(_bad):,} rows (bad date) -> {(SILVER / '_quarantine' / 'bad_postings.parquet').as_posix()}")
df = df.dropna(subset=["posted_date"]).copy()      # keep only rows with a valid date
df["posted_month"] = df["posted_date"].dt.strftime("%Y-%m")   # derive YYYY-MM (used to partition later)
df[["city", "country"]] = df["location"].apply(lambda s: pd.Series(split_location(s)))   # split location into two columns

**Step 5 - derive fields.** Internship flag, seniority tidy-up, discipline, cleaned category, and the Netherlands / Europe region tag - all as columns, so we can filter late.

**Step 5a - internship flag + seniority.** An intern is anything tagged `Internship` or whose title matches our hint words; then we tidy the (often missing) seniority level. Feeds rarely report seniority, so any remaining unknowns are defaulted to *Mid Level* - an arbitrary choice that inflates that bucket, so read the by-seniority chart as a rough sanity check, not a precise distribution.

In [ ]:
# What: flag internships (from the level or title keywords) and tidy the often-missing seniority.
# Why:  ATS feeds rarely tag 'internship', so we infer it from the title; this flag drives every later chart.

# derive is_internship, then tidy the level (ATS rarely tags level, so infer from the title)
title_is_intern = df["title"].str.lower().apply(lambda t: any(k in t for k in INT_HINTS))   # title contains a hint word?
df["is_internship"] = (df["level"].str.lower() == "internship") | title_is_intern           # internship if level says so OR title hints
df.loc[(df["level"] == "Unknown") & df["is_internship"], "level"] = "Internship"            # fill missing level for clear interns
df["level"] = df["level"].replace({"Unknown": "Mid Level"})                                  # remaining Unknowns -> a sensible default

**Step 5b - classify the discipline.** Map each title into Data Engineering / Data Science / ML-AI / Data Analytics / Software / Other. Kept as a *column* so narrowing to data roles stays a late filter.

In [ ]:
# What: label each posting: Data Engineering / Data Science / ML-AI / Analytics / Software / Other.
# Why:  kept as a COLUMN, not a download filter, so narrowing to data roles later is a cheap, reversible WHERE.

# Classify each posting into a discipline. We keep ALL rows - we only *filter* to
# data/AI/ML later, at the analysis. Bronze/silver stay complete and reusable.
def classify(title):
    t = " " + title.lower() + " "      # pad with spaces so " ai " matches the word, not "air"
    if any(k in t for k in ("data engineer", "etl ", "data pipeline", "data platform", "analytics engineer")):
        return "Data Engineering"      # most specific bucket first
    if any(k in t for k in ("machine learning", "ml engineer", " ai ", "a.i.", "artificial intelligence",
                            "deep learning", "mlops", " nlp ", " llm ", "computer vision")):
        return "ML / AI"
    if "data scien" in t:              # catches 'data science' and 'data scientist'
        return "Data Science"
    if any(k in t for k in ("data analyst", "business intelligence", "analytics", " bi ")):
        return "Data Analytics"
    if any(k in t for k in ("software", "developer", "backend", "front end", "frontend",
                            "full stack", "fullstack", " engineer", "programmer")):
        return "Software Engineering"  # broad tech bucket, checked last
    return "Other"                     # everything else (the wider market)
df["discipline"] = df["title"].apply(classify)   # add the discipline column

**Step 5c - clean the category + tag the region.** Strip ATS numeric codes from job-family labels, and tag each posting Netherlands / Europe / Remote / Other.

In [ ]:
# What: tidy messy ATS department labels and tag a region (Netherlands / Europe / Remote / Other).
# Why:  clean labels read better in charts; region as a column lets us focus on NL / EU without re-downloading.

# tidy raw ATS department labels for display (e.g. "5112 General University" -> "General University")
def clean_category(c):
    c = re.sub(r"^\s*\d+[\s\-:_.]*", "", str(c)).strip()   # strip a leading number code + its separators
    return c if c else "Other"                              # empty after cleaning -> "Other"
df["category"] = df["category"].apply(clean_category)

# tag a region so students can focus on the Netherlands / Europe (we still keep EVERY row)
NL_HINT = {"netherlands", "nederland", "amsterdam", "rotterdam", "utrecht", "eindhoven", "the hague",   # words meaning 'Netherlands'
           "den haag", "groningen", "delft", "tilburg", "nijmegen", "haarlem", "leiden"}
EU_COUNTRIES = {"Netherlands", "Germany", "United Kingdom", "Ireland", "France", "Spain",   # countries we treat as Europe
                "Belgium", "Italy", "Portugal", "Sweden", "Denmark", "Norway", "Finland",
                "Poland", "Austria", "Switzerland", "Czech Republic", "Czechia", "Romania",
                "Greece", "Hungary", "Luxembourg", "Estonia", "Lithuania", "Latvia",
                "Bulgaria", "Croatia", "Slovakia", "Slovenia"}
EU_CITY_HINT = {"berlin", "munich", "hamburg", "cologne", "frankfurt", "stuttgart",   # big EU cities (used when country is missing)
                "dusseldorf", "düsseldorf", "london", "dublin", "paris", "madrid",
                "barcelona", "lisbon", "stockholm", "copenhagen", "warsaw", "vienna",
                "zurich", "milan", "brussels"}
US_HINT = (" us ", "us-", "-us", "u.s", "(us", "us)", "usa", "united states")   # markers of a US location
US_STATES = {"alabama", "alaska", "arizona", "arkansas", "california", "colorado", "connecticut",
             "delaware", "florida", "georgia", "hawaii", "idaho", "illinois", "indiana", "iowa",
             "kansas", "kentucky", "louisiana", "maine", "maryland", "massachusetts", "michigan",
             "minnesota", "mississippi", "missouri", "montana", "nebraska", "nevada", "new hampshire",
             "new jersey", "new mexico", "new york", "north carolina", "north dakota", "ohio",
             "oklahoma", "oregon", "pennsylvania", "rhode island", "south carolina", "south dakota",
             "tennessee", "texas", "utah", "vermont", "virginia", "washington", "west virginia",
             "wisconsin", "wyoming"}   # spelled-out US states (a remote role here is still US-based)
def to_region(city, country):
    s = " " + (str(city) + " " + str(country)).lower() + " "   # pad so " us " matches as a token, not inside a word
    if any(h in s for h in NL_HINT):               # any Dutch hint -> Netherlands (checked first)
        return "Netherlands"
    if "remote" in s or "flexible" in s:           # remote / flexible...
        if any(h in s for h in US_HINT) or any(st in s for st in US_STATES):   # ...but a US-based remote role isn't 'apply from NL'
            return "Other"
        return "Remote"
    if str(country) in EU_COUNTRIES or any(h in s for h in EU_CITY_HINT):   # an EU country or city -> Europe
        return "Europe"
    return "Other"                                 # everything else (mostly US)
df["region"] = [to_region(c, k) for c, k in zip(df["city"], df["country"])]   # add the region column

**Step 6 - write partitioned silver + the bridge.** One Parquet dataset partitioned by month, plus the
posting -> category bridge for the many-to-many.

*New this week: partitioning.* Instead of one big file, we split the table into **folders, one per
month** (`posted_month=2026-05/`, ...). A query filtered by month then opens only that folder and skips
the rest, so it stays fast as the data grows. (Earlier weeks only used `PARTITION BY` for SQL window
functions - this is *storage* partitioning, and we measure its payoff in Lab 3.)

In [ ]:
# What: write silver as Parquet split into one folder per month, plus the posting -> category bridge.
# Why:  Parquet is columnar (fast); partitioning by month is our first scalability move; the bridge handles many-to-many.

postings = (df[["posting_id", "source", "company", "title", "level", "category", "discipline",   # keep only the columns we model on
                "region", "city", "country", "posted_date", "posted_month", "is_internship",
                "from_intern_pull", "url"]]
            .drop_duplicates("posting_id").reset_index(drop=True))   # one row per posting -> enforces UNIQUENESS

# write PARTITIONED silver (by month) + the bridge
sp = SILVER / "postings"
if sp.exists():                        # if a previous run left a folder...
    shutil.rmtree(sp)                  # ...delete it so we write cleanly (no leftovers)
postings.to_parquet(sp, partition_cols=["posted_month"], index=False)   # one sub-folder per posted_month

bridge = pd.DataFrame(cat_links, columns=["posting_id", "category"]).drop_duplicates()   # the many-to-many links
bridge = bridge[bridge["posting_id"].isin(set(postings["posting_id"]))]   # keep only links whose posting survived
bridge.to_parquet(SILVER / "bridge_posting_category.parquet", index=False)   # save the bridge table

print(f"silver postings: {len(postings):,} rows  ({postings['is_internship'].sum():,} internships)")
print("partitions (months):", sorted(p.name for p in sp.iterdir() if p.is_dir())[:6], "...")   # show a few month folders
postings.head()                        # peek at the result

### Check the data before you trust it (Week 3)

Before modelling, take a quick look at silver one source at a time: how many rows, how much is missing,
how many internships, and the date range. This is how you catch a source that quietly broke (zero rows,
or all dates missing) *before* it ruins the gold layer.

In [ ]:
# What: profile silver per source: row count, % unknown, internships, and date range.
# Why:  check the data BEFORE modelling, so a source that silently broke can't poison the gold layer.

# Data profiling (Week 3): shape, null rates, date coverage - per source
profile = con.execute("""
    SELECT source,                                                           -- one row per source
           COUNT(*)                                                          AS rows,                 -- how many postings
           ROUND(100.0*COUNT(*) FILTER(WHERE company = 'Unknown')/COUNT(*),1) AS pct_company_unknown, -- % missing company
           ROUND(100.0*COUNT(*) FILTER(WHERE city = 'Unknown')/COUNT(*),1)    AS pct_city_unknown,    -- % missing city
           CAST(SUM(is_internship::int) AS INTEGER)                          AS internships,          -- how many internships (whole number)
           MIN(posted_month)                                                AS earliest,             -- oldest month seen
           MAX(posted_month)                                                AS latest                -- newest month seen
    FROM postings
    GROUP BY source ORDER BY rows DESC
""").df()
profile                                # show the profile table

### ETL vs ELT - the same job, two ways (Week 3)

We just built silver the **ETL** way: pull the JSON, transform it in **Python**, then save Parquet. You
can get the exact same result the **ELT** way: load the raw JSON into the database first, then transform
it with **SQL**. Below we do one source - Greenhouse - **both ways** and check the results match exactly.

In [ ]:
# What: build the Greenhouse rows two ways - pandas (ETL) and DuckDB SQL (ELT) - and prove they match.
# Why:  same result, two styles: choose ETL for procedural / row-by-row logic, ELT when plain SQL can do it.

# ETL vs ELT on the Greenhouse feed (one representative source). Same rows, two engines.

# --- ETL: Extract -> Transform in Python (pandas) -> Load ---
etl_rows = []                          # collect the transformed rows here
for fn in BRONZE.glob("ats_greenhouse_*.json"):   # read each Greenhouse file
    for j in json.loads(fn.read_text(encoding="utf-8")).get("jobs", []):   # each job
        etl_rows.append(dict(          # transform it in Python
            posting_id="gh_" + str(j["id"]), source="greenhouse",
            company=(j.get("company_name") or "Unknown").strip(),
            title=(j.get("title") or "").strip(),
            location=(j.get("location") or {}).get("name", "Unknown"),
            posted_date=pd.to_datetime(j.get("first_published") or j.get("updated_at"),
                                       errors="coerce", utc=True)))
etl = pd.DataFrame(etl_rows)           # load into a DataFrame
etl["posted_date"] = etl["posted_date"].dt.tz_localize(None)   # drop timezone so both sides match

# --- ELT: Extract + Load raw -> Transform inside the engine (DuckDB SQL) ---
gh_glob = (BRONZE / "ats_greenhouse_*.json").as_posix()   # a wildcard path DuckDB can read directly
elt_sql = f"""
    WITH raw AS (
        SELECT UNNEST(jobs) AS j                                          -- explode the 'jobs' array into rows
        FROM read_json_auto('{gh_glob}', union_by_name=true, maximum_object_size=200000000)   -- read all files at once
    )
    SELECT 'gh_' || j.id::VARCHAR                        AS posting_id,    -- same id rule as the ETL side
           'greenhouse'                                  AS source,
           COALESCE(NULLIF(TRIM(j.company_name), ''), 'Unknown') AS company,   -- trim; blank -> Unknown
           COALESCE(TRIM(j.title), '')                   AS title,
           COALESCE(j.location.name, 'Unknown')          AS location,
           TRY_CAST(COALESCE(j.first_published, j.updated_at) AS TIMESTAMP) AS posted_date   -- safe date cast
    FROM raw
"""
elt = con.execute(elt_sql).df()        # run the SQL and pull the result into a DataFrame

# the ELT idiom in one line: land the transformed result straight to columnar Parquet
con.execute(f"COPY ({elt_sql}) TO '{(SILVER / '_elt_greenhouse.parquet').as_posix()}' (FORMAT PARQUET)")   # write result to Parquet

print(f"ETL (pandas) rows : {len(etl):,}")
print(f"ELT (DuckDB) rows : {len(elt):,}")
print("same posting_id set:", set(etl.posting_id) == set(elt.posting_id))   # do both produce the same ids?
print("same columns       :", list(etl.columns) == list(elt.columns))       # and the same columns?
assert len(etl) == len(elt) and set(etl.posting_id) == set(elt.posting_id), "ETL and ELT disagree!"   # fail if they differ
print("\nSame data, two engines. Pick ETL for procedural / ML / row-by-row logic;")
print("pick ELT when the transform is set-based SQL the warehouse itself can run.")

> **Looking back at Weeks 2 and 3 (profiling, quality, storage).**
> - **Profile before you clean (Week 3):** check each source's row count, missing values, types and date range *before* you trust it - the table above is exactly that check. Cleaning blindly hides rows you accidentally drop.
> - **The six data-quality questions (Week 3):** is the data complete, unique, valid, consistent, accurate, and up to date? Here `drop_duplicates("posting_id")` keeps it **unique**, and the date check keeps it **valid**. Rows with an unreadable date go to a **quarantine** folder (above), not deleted - so nothing disappears quietly.
> - **Parquet is columnar and compressed (Week 2):** silver is Parquet, so a query that needs two columns reads only those two. That is the Week 2 "CSV vs Parquet" lesson, now inside your own pipeline.
> - **Delta Lake (Week 2):** our "delete the folder and rewrite it" is a basic version of a safe write. **Delta Lake** would do the same step with real safety guarantees, schema checks, and time travel (reading an older version of the table) - without managing folders by hand.

> **Challenge: Why?** We split the data into folders by **month**, not by **company**. Which questions get faster, and which do not?
> <details><summary>Answer</summary>Splitting by month speeds up anything filtered by time ("postings
> since March", "this quarter") - the database skips every other month's folder. It does *nothing* for
> "all postings at Adyen", which still has to look in every month. Split on the column you filter by
> most often; here that is time, because we re-run the pipeline every week.</details>

## Lab 2 - Gold: design the star schema yourself

This is the real test of what you learned. Nobody hands you the tables - you build them with Kimball's
**four-step recipe**:

1. **Pick the process** - companies post jobs.
2. **Pick the grain** - *one row per posting.* Decide this first, before anything else.
3. **Pick the dimensions** - the who / what / where / when: company, location, level, job family, discipline, date.
4. **Pick the facts** - the number you measure at that grain. One posting is one event, so the number we
   add up is simply a **count of postings**.

> **Challenge: Why?** Why is *one row per posting* the right grain - and what breaks if you choose
> *one row per company* instead?
> <details><summary>Answer</summary>The questions we care about live at the posting level: which role,
> which city, which level, which month. "One row per company" throws all that away - you could never
> break a company down by city or seniority again. Always choose the *most detailed* grain your questions
> need. You can always add things up later; you can never split them back apart.</details>

> **Looking back at Weeks 1 and 4 (how the model fits in).**
> - **Normalization / 3NF (Week 1):** the systems that *run* a business are normalized - data is split across many tables to avoid update mistakes. A warehouse does the opposite on purpose: we **denormalize** into a star so analysts write fewer joins. The **star vs snowflake** choice below is just "how far do we normalize this one dimension?".
> - **Kimball, Inmon, Data Vault (Week 4):** we build a **Kimball** star (simple and fast for reporting). *Inmon* would build one big normalized warehouse first; *Data Vault* is built for auditing and tracking history. Same data - different starting philosophy.

**Step 1 - load silver and build the dimensions.** A small helper that gives each row an ID number, then one `dim_*` table for each descriptive attribute (company, location, level, job family, discipline).

In [ ]:
# What: load silver and build one dimension table per descriptive attribute, each with an ID key.
# Why:  dimensions hold the who / what / where; surrogate IDs keep the fact table small and stable (Kimball).

post = pd.read_parquet(SILVER / "postings")   # load the clean silver table (all months) into one DataFrame

def make_dim(frame, cols, key):        # helper: turn some columns into a dimension table
    d = frame[cols].drop_duplicates().sort_values(cols).reset_index(drop=True)   # unique values, sorted, fresh index
    d.insert(0, key, range(1, len(d) + 1))   # add a surrogate ID column (1, 2, 3, ...)
    return d                            # hand back the dimension

dim_company    = make_dim(post, ["company"], "company_key")                       # WHO posted the job
dim_location   = make_dim(post, ["region", "country", "city"], "location_key")    # WHERE: region -> country -> city
dim_level      = make_dim(post, ["level"], "level_key")                           # seniority
dim_category   = make_dim(post, ["category"], "category_key")                     # job family / department
dim_discipline = make_dim(post, ["discipline"], "discipline_key")                 # our data / AI / etc. label

**Step 2 - the date dimension.** Derived from `posted_month`, with year / quarter for easy roll-ups.

In [ ]:
# What: build the date dimension from posted_month, with year and quarter.
# Why:  a shared date dimension lets us roll up by month or quarter and line several facts up on one timeline.

dim_date = post[["posted_month"]].drop_duplicates().sort_values("posted_month").reset_index(drop=True)   # the unique months
dim_date["year"] = dim_date["posted_month"].str[:4].astype(int)     # first 4 characters -> year
dim_date["month"] = dim_date["posted_month"].str[5:7].astype(int)   # characters 6-7 -> month number (1-12)
dim_date["quarter"] = ((dim_date["month"] - 1) // 3 + 1)            # month number -> quarter (1-4)
dim_date.insert(0, "date_key", range(1, len(dim_date) + 1))         # surrogate ID for each month

**Step 3 - build the fact table and check it.** Attach each posting's dimension IDs, then check every ID matched (no broken links) - the Week 4 surrogate-key safety check.

In [ ]:
# What: build fact_posting (dimension keys + the internship flag) and check every key matched.
# Why:  the grain is one row per posting; the referential-integrity check catches broken joins straight away.

# fact_posting: keys + the is_internship flag; the measure is the row itself (count)
fact_posting = (post
    .merge(dim_company, on="company")                               # swap company text -> company_key
    .merge(dim_location, on=["region", "country", "city"])          # swap location -> location_key
    .merge(dim_level, on="level")                                   # swap level -> level_key
    .merge(dim_category, on="category")                             # swap category -> category_key
    .merge(dim_discipline, on="discipline")                         # swap discipline -> discipline_key
    .merge(dim_date[["date_key", "posted_month"]], on="posted_month")   # swap month -> date_key
    [["posting_id", "company_key", "location_key", "level_key", "category_key", "discipline_key", "date_key",
      "is_internship", "from_intern_pull"]]   # keys + the internship flag + the sampling-provenance flag
    .reset_index(drop=True))

# referential integrity (the P4 guardrail from Week 4)
for k in ["company_key", "location_key", "level_key", "category_key", "discipline_key", "date_key"]:
    assert fact_posting[k].notna().all(), f"unmatched {k}"          # every row must have matched a dimension (no broken links)
print("Referential integrity OK.")
print(f"fact_posting: {len(fact_posting):,} rows | dims:",          # how big the fact is, plus each dimension's size
      {"company": len(dim_company), "location": len(dim_location), "level": len(dim_level),
       "category": len(dim_category), "discipline": len(dim_discipline), "date": len(dim_date)})
# show BOTH sides of the star: a dimension (the descriptive 'who') and the fact (the measure)
print("\ndim_company sample - the descriptive 'who' side of the star:")
display(dim_company.head())
print("fact_posting sample - keys + the measure, the centre of the star:")
fact_posting.head()                    # peek at the star's fact table

### Star vs snowflake - a real choice for `dim_location`

Our `dim_location` holds a **region -> country -> city** order. There are two ways to store that:

- **Star (what we built):** one flat `dim_location` table. The fact table joins to it **once**. Simple
  and fast, with a little repeated text (the word "Europe" shows up on every European city row).
- **Snowflake:** split it into a separate `dim_region` table linked to `dim_location`. No repeated text,
  but every query now needs an **extra join**.

Below we run both. They give the **same answer**, so for reporting we pick the star and accept a little repetition.

In [ ]:
# What: answer the same question with a flat dimension (star) and a normalised one (snowflake).
# Why:  identical answer, but star = one join and snowflake = two - which is why we default to the star for reporting.

# STAR: the flat dimension - one join from the fact
star_q = con.execute("""
    SELECT l.region, COUNT(*) AS postings                                       -- count postings per region
    FROM fact_posting f JOIN dim_location l ON l.location_key = f.location_key   -- ONE join: fact -> flat dimension
    GROUP BY l.region ORDER BY postings DESC
""").df()

# SNOWFLAKE: normalize region out into its own table, then join *through* it
con.execute("CREATE OR REPLACE TABLE dim_region AS "                            # pull the distinct regions into their own tiny table
            "SELECT ROW_NUMBER() OVER (ORDER BY region) AS region_key, region "  # give each region an id
            "FROM (SELECT DISTINCT region FROM dim_location)")
con.execute("""CREATE OR REPLACE TABLE dim_location_snow AS
    SELECT l.location_key, r.region_key, l.country, l.city                       -- location now points to region_key, not the text
    FROM dim_location l JOIN dim_region r ON r.region = l.region""")
snow_q = con.execute("""
    SELECT r.region, COUNT(*) AS postings
    FROM fact_posting f
    JOIN dim_location_snow ls ON ls.location_key = f.location_key                -- join 1: fact -> location
    JOIN dim_region r          ON r.region_key   = ls.region_key                 -- join 2: location -> region (the extra hop)
    GROUP BY r.region ORDER BY postings DESC
""").df()

print("dim_location rows (star):", len(dim_location))
print("same answer both ways    :", star_q.equals(snow_q))                       # identical result...
print("star = 1 join, snowflake = 2 joins for the identical result -> default to star for BI.")   # ...for less work with the star
star_q

### Three kinds of numbers: additive, semi-additive, non-additive (the Week 4 trap)

- **Additive** - a **count of postings**. You can safely add it up across anything: by company, by month, by city.
- **Semi-additive** - a number that is a *snapshot in time*, like how many roles a company has open at the
  end of a month. You can add it across companies, but **not** across months (a role open for three months
  would get counted three times). True semi-additive data needs a *closed* date to know what is still open,
  but our feeds only give a *posted* date. So `fact_company_month` below counts **postings posted each
  month** - which is actually an **additive** measure - and we use it only to show the company x month grain
  and the drill-across. Treat semi-additive here as a *concept*, the same way we handle SCD below.
- **Non-additive** - **internship share** = internships / all jobs. This is a fraction. You **cannot** add
  fractions up or average them naively across groups.

> **A trap we have to avoid (know your data, Week 3).** We *deliberately* over-collected internships
> (a dedicated internship pull from The Muse). So a company we only ever saw in that pull would have an
> internship share of a fake **100%** - not because it only hires interns, but because we never fetched
> its other jobs. A share is only fair when the **denominator** is complete. So we tag every row with
> `from_intern_pull` and compute the company share **only over the fully-pulled sources** (the ATS feeds,
> Arbeitnow, and the general Muse sample). The counts elsewhere still use every row.

In [ ]:
# What: compute internship share per company (interns divided by all postings).
# Why:  this is a NON-additive measure: a ratio you must weight by the denominator, never just average.
#       NOTE: we computed share only over FULLY-pulled sources (NOT from_intern_pull) so the denominator is fair -
#       a company we only fetched internships for would otherwise show a fake 100%.

con.register("fact_posting", fact_posting)     # make the pandas tables visible to SQL, by name
con.register("dim_company", dim_company)
con.register("dim_level", dim_level)
con.register("dim_date", dim_date)

# non-additive done RIGHT vs WRONG: internship share by company (top 10 by volume)
share = con.execute("""
    SELECT c.company,
           COUNT(*)                                          AS total_postings,                  -- all postings
           SUM(CASE WHEN f.is_internship THEN 1 ELSE 0 END)  AS intern_postings,                 -- just the internships
           ROUND(100.0 * SUM(CASE WHEN f.is_internship THEN 1 ELSE 0 END) / COUNT(*), 1) AS intern_share_pct   -- share = interns / all
    FROM fact_posting f
    JOIN dim_company c ON c.company_key = f.company_key
    WHERE NOT f.from_intern_pull                                                                 -- fair denominator: drop internship-only pulls
    GROUP BY c.company
    HAVING COUNT(*) >= 20                                                                        -- ignore tiny companies (a noisy %)
    ORDER BY intern_share_pct DESC, total_postings DESC
    LIMIT 10
""").df()
share

> **Challenge: Why?** A colleague works out the overall internship share with `AVG(intern_share_pct)`
> across the companies above. Why is that wrong, and what is correct?
> <details><summary>Answer</summary>Averaging the percentages gives every company equal weight, no matter
> its size. A tiny company with 1 intern out of 2 roles (50%) would count as much as a giant with 500 out
> of 5000 (10%). The correct overall share is **total interns / total jobs**, so bigger companies count
> more. This is the same mistake as averaging average income in Week 4.</details>

### Tracking company changes over time (Slowly Changing Dimensions)

Companies rename and change size - a rebrand, a takeover. One live download is only a single snapshot in
time, so (just like Week 4) we show the *structure* with a clear before-and-after example. **Type 1**
overwrites the old value, so history is lost. **Type 2** keeps both the old and new rows, each with a
"valid from / valid to" window.

In [ ]:
# What: show a company rename as an SCD Type-2 before/after (valid_from / valid_to / is_current).
# Why:  keeps history correct: a 2025 posting still maps to the company's 2025 name, not today's.

# worked example: a company rebrands between two pulls
scd2 = pd.DataFrame([
    {"company_sk": 1, "company_code": "C-1001", "company_name": "Acme Robotics",          # OLD row: the 2025 version
     "size_band": "201-500", "valid_from": "2025-01-01", "valid_to": "2025-12-31", "is_current": False},   # closed at end of 2025
    {"company_sk": 2, "company_code": "C-1001", "company_name": "Acme AI",                # NEW row: same company (same code), renamed
     "size_band": "501-1000", "valid_from": "2026-01-01", "valid_to": None, "is_current": True},   # valid_to = None means still current
])
print("SCD Type 1 would keep only the current row - the 2025 name 'Acme Robotics' is lost.")
print("SCD Type 2 keeps both, so 'what was C-1001 called in mid-2025?' is still answerable:")
scd2

> **Challenge: Why?** Why use SCD **Type 2** on `dim_company` instead of Type 1 (overwrite)?
> <details><summary>Answer</summary>A posting from mid-2025 was made under the company's *old* name and
> size. With Type 1 you would relabel all that old data with today's values - quietly rewriting the past,
> so a "postings by company size at the time" report would be wrong. Type 2 keeps the old rows, so the
> past stays correct.</details>

### A shared dimension, used by two facts (drill-across)

`dim_date` and `dim_company` are **shared dimensions**: more than one fact table uses them. We add a
second fact table, `fact_company_month` (how many postings each company has per month). Because it shares
`dim_company` and `dim_date` with `fact_posting`, we can line the two up on the same timeline and compare them.

In [ ]:
# What: build a second fact (postings per company per month) and compare it to the first on one timeline.
# Why:  dim_date and dim_company are shared (conformed) - that shared key is what lets us 'drill across' two facts.

# second fact: postings per company per month (an ADDITIVE count - see the note above on semi-additive)
fact_company_month = con.execute("""
    SELECT f.company_key, f.date_key,                                                  -- grain: one row per company per month
           COUNT(*)                                          AS postings_that_month,         -- postings that month
           SUM(CASE WHEN f.is_internship THEN 1 ELSE 0 END)  AS internships_that_month       -- of which, internships
    FROM fact_posting f
    GROUP BY f.company_key, f.date_key
""").df()
con.register("fact_company_month", fact_company_month)   # register the new fact for SQL

# drill-across: postings vs internships per month, on the shared dim_date
drill = con.execute("""
    SELECT d.posted_month,
           SUM(s.postings_that_month)    AS postings,                  -- total postings per month
           SUM(s.internships_that_month) AS internships               -- total internships per month
    FROM fact_company_month s
    JOIN dim_date d ON d.date_key = s.date_key                   -- the SHARED date dimension makes this join possible
    GROUP BY d.posted_month
    ORDER BY d.posted_month
""").df()
print(f"fact_company_month: {len(fact_company_month):,} (company x month) rows")
drill.tail(8)                          # show the last 8 months

### Save the gold layer

Save the star to `week5/data/gold` so that reporting tools, the semantic view, and any future work can
read a clean, finished model without re-running the whole pipeline.

In [ ]:
# What: save every dimension and fact to the gold folder as Parquet.
# Why:  a clean, finished model that BI tools or future work can read without re-running the whole pipeline.

gold = {                               # everything we want to persist: name -> table
    "dim_company": dim_company, "dim_location": dim_location, "dim_level": dim_level,
    "dim_category": dim_category, "dim_discipline": dim_discipline, "dim_date": dim_date,
    "fact_posting": fact_posting, "fact_company_month": fact_company_month,
}
for name, frame in gold.items():       # write each table...
    frame.to_parquet(GOLD / f"{name}.parquet", index=False)   # ...as its own Parquet file in the gold folder
print("Gold ready:", sorted(p.name for p in GOLD.glob("*.parquet")))   # confirm what was written

## Lab 3 - Scalability you can measure

Three things you can now *measure*, instead of just claiming.

### 3a. Partition pruning
Silver is split into one folder per month. A query filtered by month opens only the matching folder and
skips all the others. We show this two ways: first the **deterministic** part - how many folders/rows the
filter lets us skip - then a timing (which barely moves at 5k rows, but is the thing that pays off at GBs).

In [ ]:
# What: show partition pruning two ways - how much is SKIPPED (deterministic) and the time (noisy here).
# Why:  the real win of partitioning is reading fewer folders/rows; at this small size the timing barely moves.

import time as _t                       # _t.perf_counter() is a precise stopwatch
PART = (SILVER / "postings").as_posix()  # path to the partitioned silver folder
month_dirs = [p for p in (SILVER / "postings").iterdir() if p.is_dir()]   # one folder per month
latest_month = sorted(d.name.split("=")[1] for d in month_dirs)[-1]       # newest month folder, e.g. '2026-06'

# 1) DETERMINISTIC: how much does a one-month filter let us skip?
rows_total  = con.execute(f"SELECT COUNT(*) FROM read_parquet('{PART}/**/*.parquet', hive_partitioning=true)").fetchone()[0]
rows_pruned = con.execute(f"SELECT COUNT(*) FROM read_parquet('{PART}/**/*.parquet', hive_partitioning=true) WHERE posted_month = '{latest_month}'").fetchone()[0]
print(f"partitioning: a filter on month opens 1 of {len(month_dirs)} folders")
print(f"  pruned (month = {latest_month}): reads ~{rows_pruned:,} rows from 1 folder")
print(f"  full scan (all months)        : reads {rows_total:,} rows from {len(month_dirs)} folders\n")

# 2) TIMING: real, but tiny and noisy at this size (the point is the skipping above, which pays off at GBs)
def timeit(label, sql, runs=5):          # run a query a few times and report the median time
    ts = []                              # collect the timings
    for _ in range(runs):                # repeat to smooth out noise
        s = _t.perf_counter(); con.execute(sql).fetchall(); ts.append(_t.perf_counter() - s)   # time one run
    print(f"{label:>34}: median {sorted(ts)[runs//2]*1000:.1f} ms")   # median time in milliseconds

q_pruned = f"SELECT COUNT(*) FROM read_parquet('{PART}/**/*.parquet', hive_partitioning=true) WHERE posted_month = '{latest_month}'"   # filter to ONE month
q_full   = f"SELECT COUNT(*) FROM read_parquet('{PART}/**/*.parquet', hive_partitioning=true)"   # scan ALL months
timeit("partition-pruned (one month)", q_pruned)   # opens just one folder
timeit("full scan (all months)", q_full)           # opens every folder
print("(at 5k rows the times are nearly equal - even noisy; the win is skipping folders, which scales to GBs.)")

### 3b. Safe, repeatable loads
A real pipeline runs again and again - say, every week. Running it again must **never** create duplicate
rows. Here we copy that idea: keep only the postings we do not already have, and add those by their key.
Run the cell twice - the second run adds **zero** rows.

In [ ]:
# What: take a fresh pull and keep only the postings we do not already have.
# Why:  a real pipeline re-runs weekly; this makes re-running safe - the second run adds zero rows (idempotent).

# simulate "what we already have" vs "a fresh pull" using posting_id as the key
existing_ids = set(pd.read_parquet(SILVER / "postings")["posting_id"])   # the ids already saved in silver
fresh = post  # in a real run this is a new API pull; here we re-use the same frame

new_rows = fresh[~fresh["posting_id"].isin(existing_ids)]   # rows whose id we have NOT seen before
print(f"already have: {len(existing_ids):,} | fresh pull: {len(fresh):,} | genuinely new: {len(new_rows):,}")
print("-> an idempotent load appends only the new rows, so re-running is safe (0 new on a repeat).")

> **Challenge: Why?** At **100x** this data (say 5 GB of postings a day), what changes - and when would
> you stop using one DuckDB machine and move to a cluster?
> <details><summary>Answer</summary>Columnar files plus partitioning take you a long way: one big machine
> handles tens to a few hundred GB comfortably. You switch to many machines (Spark, Databricks, a cloud
> warehouse) when a single day's data no longer fits in memory for big joins, when many people query at
> once, or when the pipeline must finish in minutes over years of history. Until then, one bigger machine
> plus partitioning is cheaper and simpler.</details>

## Lab 4 - Use it: answer the question

A warehouse only matters if people use it. We build a small **semantic view** (a query with friendly,
business-style names) and then chart the answer to our question.

In [ ]:
# What: create vw_intern_market - a view with friendly, business names over the star.
# Why:  analysts query readable names, not raw keys; this is the semantic layer that answers our question.

con.execute("""
    CREATE OR REPLACE VIEW vw_intern_market AS                         -- a saved query (the 'semantic layer')
    SELECT c.company          AS company,                              -- turn each key back into a readable label
           l.city             AS city,
           l.country          AS country,
           l.region           AS region,
           lv.level           AS seniority,
           cat.category       AS job_family,
           dis.discipline     AS discipline,
           d.posted_month     AS month,
           f.is_internship    AS is_internship
    FROM fact_posting f                                                -- the fact in the centre...
    JOIN dim_company c    ON c.company_key   = f.company_key           -- ...joined out to each dimension (the star)
    JOIN dim_location l   ON l.location_key  = f.location_key
    JOIN dim_level lv     ON lv.level_key    = f.level_key
    JOIN dim_category cat  ON cat.category_key  = f.category_key
    JOIN dim_discipline dis ON dis.discipline_key = f.discipline_key
    JOIN dim_date d       ON d.date_key      = f.date_key
""")
con.execute("SELECT COUNT(*) FROM vw_intern_market").df()              # quick check: how many rows the view exposes

### Act 1 - The job market overall

Start wide: what does the whole market in this snapshot look like, before we zoom in?

Heads-up: the biggest names in these charts (CVS, TikTok, Walmart, ...) come from The Muse's
**global sample** - we keep it on purpose as the "wider market" to compare against. The
**Netherlands / Europe focus comes in Act 3.**

In [ ]:
# What: plot all postings vs internships per month - over the recent, data-rich months only.
# Why:  ~85% of postings fall in the last year, so the long, sparse 2019-2024 tail just clutters the
#        chart; we focus on the most recent 18 months, where the trend actually is.

recent = drill.tail(18)                                                # last 18 months (older ones are too sparse to read)
fig, ax = plt.subplots(figsize=(9, 3.6))                              # one wide chart
ax.plot(recent["posted_month"], recent["postings"], marker="o", color=SLATE, label="all postings")    # whole-market line
ax.plot(recent["posted_month"], recent["internships"], marker="s", color=TEAL, label="internships")   # internships line
ax.set_title("The job market vs the intern market, by month (last 18 months)", color=SLATE)
ax.tick_params(axis="x", rotation=45); ax.legend()                   # rotate the month labels; show the legend
plt.tight_layout(); plt.show()
print(f"showing {recent['posted_month'].iloc[0]} -> {recent['posted_month'].iloc[-1]}  "
      f"(earlier months exist in the data but are too sparse to plot)")

> **Reading this honestly.** We *deliberately* over-collected internships (a dedicated internship pull),
> so the internships line is **boosted** - especially in some early-year months where most of those
> postings landed. Read it as "interns are a small, intentionally-sampled slice of the market", not as
> raw seasonality. The fair *ratio* (internship share) is handled separately in Lab 2.

In [ ]:
# What: internship share by seniority level.
# Why:  a sanity check - interns should dominate the 'Internship' level if our flag is working.

# internship share by seniority level (sanity: interns should dominate the "Internship" level)
fig, ax = plt.subplots(figsize=(7, 3.6))
g = con.execute("""
    SELECT seniority, ROUND(100.0*SUM(CASE WHEN is_internship THEN 1 ELSE 0 END)/COUNT(*),1) AS intern_pct   -- % of this level that are internships
    FROM vw_intern_market GROUP BY seniority ORDER BY intern_pct DESC
""").df()
ax.barh(g["seniority"][::-1], g["intern_pct"][::-1], color=TEAL)      # horizontal bars, biggest on top
ax.set_title("Internship share by seniority level (%)", color=SLATE)
plt.tight_layout(); plt.show()

In [ ]:
# What: companies with the highest internship share (at least 20 postings).
# Why:  share, not raw count, so big and small employers compare fairly - the non-additive measure in action.

# which companies are most intern-friendly (share, min 20 postings)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(share["company"][::-1], share["intern_share_pct"][::-1], color=TEAL)   # reuse the 'share' table built earlier
ax.set_title("Most intern-friendly companies (internship share %, fully-pulled sources, >= 20)", color=SLATE)
plt.tight_layout(); plt.show()

### Act 2 - Zoom in on data, data-science, and ML/AI roles

We collected *every* kind of job, but this is a data programme. Because we saved `discipline` as a
**column** (instead of filtering it out while downloading), narrowing to data and AI roles is a single
`WHERE` line - and bronze and silver stay complete for any other use later.

> **Challenge: Why?** Why label every role and then filter, instead of only downloading data/AI roles in the first place?
> <details><summary>Answer</summary>Filtering at download time throws data away for good - you could never
> ask "how does data hiring compare to the rest of the market?" or switch to another field without
> downloading everything again. A `discipline` column keeps all the raw data and makes the focus a cheap,
> undoable `WHERE`. Collecting broadly and filtering late keeps the warehouse flexible.</details>

In [ ]:
# What: internships by discipline, with data / AI / ML highlighted.
# Why:  this is a data programme, so we show how many internships fall in our fields versus the rest.

# internships by discipline (data/AI/ML in teal, everything else greyed out)
DATA_AI_ML = ("Data Engineering", "Data Science", "ML / AI", "Data Analytics")   # the disciplines we highlight
g = con.execute("""
    SELECT discipline, COUNT(*) AS interns
    FROM vw_intern_market WHERE is_internship                          -- internships only
    GROUP BY discipline ORDER BY interns DESC
""").df()
colors = [TEAL if d in DATA_AI_ML else "#CBD5E1" for d in g["discipline"]]   # teal for our fields, grey for the rest
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(g["discipline"][::-1], g["interns"][::-1], color=colors[::-1])   # reverse so the biggest sits on top
ax.set_title("Internships by discipline (data / AI / ML highlighted)", color=SLATE)
plt.tight_layout(); plt.show()

In [ ]:
# What: top companies hiring data / AI / ML interns (global view).
# Why:  who to look at for a data internship, before we narrow down to the Netherlands and Europe.

# which companies hire the most DATA / AI / ML interns
g = con.execute("""
    SELECT company, COUNT(*) AS interns
    FROM vw_intern_market
    WHERE is_internship AND discipline IN ('Data Engineering','Data Science','ML / AI','Data Analytics')   -- data internships only
    GROUP BY company ORDER BY interns DESC LIMIT 10
""").df()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(g["company"][::-1], g["interns"][::-1], color=TEAL)
ax.set_title("Top companies hiring data / AI / ML interns", color=SLATE); ax.set_xlabel("intern postings")
plt.tight_layout(); plt.show()

In [ ]:
# What: top job families for internships (using the cleaned labels).
# Why:  shows what kinds of roles interns actually do, not just the discipline buckets.

# top job families for internships (cleaned ATS labels)
fig, ax = plt.subplots(figsize=(8, 4))
g = con.execute("""
    SELECT job_family, COUNT(*) AS interns
    FROM vw_intern_market
    WHERE is_internship AND job_family NOT IN ('Other', 'Unknown')     -- drop the catch-all buckets
    GROUP BY job_family ORDER BY interns DESC LIMIT 10
""").df()
ax.barh(g["job_family"][::-1], g["interns"][::-1], color=TEAL)
ax.set_title("Top job families hiring interns", color=SLATE); ax.set_xlabel("intern postings")
plt.tight_layout(); plt.show()

> **Reminder from Week 3: window functions.** "Top 3 within each group" is a job for a window function,
> not a complicated self-join. Below, `ROW_NUMBER() OVER (PARTITION BY discipline ORDER BY ... DESC)`
> ranks job families *inside each discipline* and keeps the top 3 - the same `ROW_NUMBER` you used to
> remove duplicates in Week 3, now used to rank.

In [ ]:
# What: top 3 internship job families WITHIN each discipline, using ROW_NUMBER.
# Why:  'top-N per group' is a window-function job, not a self-join - the Week-3 pattern, here used to rank.

# Window function: top 3 internship job families WITHIN each discipline
top_families = con.execute("""
    WITH fam AS (                                                      -- step 1: count intern postings per (discipline, family)
        SELECT dis.discipline,
               cat.category AS job_family,
               COUNT(*) FILTER (WHERE f.is_internship) AS intern_postings   -- count only the internships
        FROM fact_posting f
        JOIN dim_discipline dis ON dis.discipline_key = f.discipline_key
        JOIN dim_category  cat ON cat.category_key   = f.category_key
        GROUP BY dis.discipline, cat.category
    )
    SELECT discipline, job_family, intern_postings
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY discipline           -- step 2: rank families WITHIN each discipline
                                     ORDER BY intern_postings DESC, job_family) AS rn   -- biggest first
        FROM fam
    )
    WHERE rn <= 3 AND intern_postings > 0                              -- step 3: keep the top 3 per discipline
    ORDER BY discipline, intern_postings DESC
""").df()
top_families

> **Why the buckets look generic.** `job_family` here is the **raw ATS department** label (e.g. "Data and
> Analytics", "Other"), not our tidy discipline - and the data-internship counts are small, so generic
> departments dominate. The point of this cell is the **technique** (`ROW_NUMBER` to rank within a group),
> which works the same on a bigger, cleaner feed.

### Act 3 - Where can *you* actually apply? Netherlands and Europe

You are an InHolland student, so the real question is "where can I apply?". Because we saved a **region**
label (Netherlands / Europe / Remote / Other) as a column, we can turn the whole product toward Europe,
with the Netherlands highlighted, using three short queries. (Note: free job data has relatively few
Netherlands-only roles, so the Dutch numbers here are small; you could add another free source later for
more coverage.)

In [ ]:
# What: internships by region, with the Netherlands highlighted.
# Why:  answers 'where are the internships?' - the start of the where-can-you-apply view.

# Where the internships are: by region (Netherlands highlighted)
g = con.execute("""
    SELECT region, COUNT(*) AS interns
    FROM vw_intern_market WHERE is_internship                          -- internships only
    GROUP BY region ORDER BY interns DESC
""").df()
cmap = {"Netherlands": TEAL, "Europe": SLATE, "Remote": "#5EAACD", "Other": "#CBD5E1"}   # one colour per region
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar(g["region"], g["interns"], color=[cmap.get(r, "#CBD5E1") for r in g["region"]])   # NL stands out in teal
ax.set_title("Where the internships are (Netherlands highlighted)", color=SLATE)
ax.set_ylabel("intern postings")
plt.tight_layout(); plt.show()

In [ ]:
# What: top European cities for internships, with Dutch cities in teal.
# Why:  concrete places to apply, with the Netherlands highlighted so students see local options.

# Top European cities for internships (Dutch cities highlighted in teal)
g = con.execute("""
    SELECT city, region, COUNT(*) AS interns
    FROM vw_intern_market
    WHERE is_internship AND region IN ('Netherlands', 'Europe')        -- NL + the rest of Europe
      AND city NOT IN ('Unknown', 'Flexible / Remote')                 -- drop the non-cities
    GROUP BY city, region ORDER BY interns DESC LIMIT 12
""").df()
fig, ax = plt.subplots(figsize=(8, 4))
colors = [TEAL if r == "Netherlands" else SLATE for r in g["region"][::-1]]   # Dutch cities teal, others slate
ax.barh(g["city"][::-1], g["interns"][::-1], color=colors)
ax.set_title("Top European cities for internships (NL in teal)", color=SLATE)
ax.set_xlabel("intern postings")
plt.tight_layout(); plt.show()

In [ ]:
# What: European and remote companies hiring data / AI / ML interns.
# Why:  the practical shortlist - data roles you could realistically apply to from the Netherlands.

# European (+ remote) companies hiring DATA / AI / ML interns - your shortlist
g = con.execute("""
    SELECT company, COUNT(*) AS interns
    FROM vw_intern_market
    WHERE is_internship
      AND region IN ('Netherlands', 'Europe', 'Remote')               -- places you can realistically apply from NL
      AND discipline IN ('Data Engineering', 'Data Science', 'ML / AI', 'Data Analytics')   -- data roles only
    GROUP BY company ORDER BY interns DESC LIMIT 10
""").df()
if len(g):                             # only draw if we actually found some
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(g["company"][::-1], g["interns"][::-1], color=TEAL)
    ax.set_title("EU + remote companies hiring data / AI / ML interns", color=SLATE)
    ax.set_xlabel("intern postings")
    plt.tight_layout(); plt.show()
else:                                  # otherwise say why it's empty
    print("No EU/remote data/AI/ML internships in this snapshot - try widening the region filter.")
g                                      # show the table as well

### Your shortlist with links - real internships you can actually open

A data product is only useful if it leads to action. We kept the original posting **`url`** all the way
through silver, so here are the **actual data / AI / data-science (and software) internships** in the
Netherlands, the rest of Europe, and remote - each title is a **clickable link** straight to the posting.
(It is a live snapshot, so re-running the notebook refreshes the list.)

In [ ]:
# What: print clickable links to REAL data / AI / data-science (+ software) internships you can apply to.
# Why:  the whole point of the pipeline is a usable answer - here are the actual postings, with their URLs.

from IPython.display import HTML, display
from html import escape as esc                         # keep odd characters in titles from breaking the HTML

REF = con.execute("""
    SELECT region, discipline, company, title, city, url
    FROM post                                          -- silver still has the original posting link
    WHERE is_internship
      AND discipline IN ('Data Engineering','Data Science','ML / AI','Data Analytics','Software Engineering')
      AND region IN ('Netherlands','Europe','Remote')  -- places you can realistically apply to from NL
      AND url LIKE 'http%'                             -- only rows with a real, clickable link
    ORDER BY CASE region WHEN 'Netherlands' THEN 0 WHEN 'Europe' THEN 1 ELSE 2 END,  -- NL first
             discipline, company
""").df()

print(f"{len(REF)} data / AI / data-science (+ software) internships in NL / Europe / Remote, with a link:\n")
html = ["<div style=\"font-family:'Segoe UI',sans-serif;font-size:13px;line-height:1.5;\">"]
for region, grp in REF.groupby("region", sort=False):                                # one block per region
    html.append(f"<div style='margin-top:10px;font-weight:700;color:#156082;'>{esc(region)} &middot; {len(grp)} roles</div>")
    for _, r in grp.iterrows():
        html.append("<div style='margin:2px 0 2px 12px;'>"
                    f"<span style='color:#94A3B8;'>[{esc(str(r.discipline))}]</span> "
                    f"<b>{esc(str(r.company))}</b> &mdash; "
                    f"<a href=\"{esc(str(r.url))}\" target=\"_blank\">{esc(str(r.title))}</a> "
                    f"<span style='color:#64748B;'>&middot; {esc(str(r.city))}</span></div>")
html.append("</div>")
display(HTML("".join(html)))

## Concept coverage matrix - all of Weeks 1-4, in one build

Use this as the checklist for the capstone: every concept from the course, and where it shows up here.

| Week | Concept | Where in this capstone |
|---|---|---|
| 1 | OLTP vs OLAP | Big-picture diagram - the sources are operational (OLTP), the gold star is analytical (OLAP) |
| 1 | 3-tier architecture | Big-picture diagram - the medallion is source -> storage -> serving |
| 1 | ACID | Big-picture diagram; Delta Lake note in Lab 1 |
| 1 | Normalization / 3NF | Lab 2 recall + the star-vs-snowflake step |
| 1 | Row vs columnar storage | Big-picture diagram; Parquet in silver/gold; Lab 3 benchmark |
| 1 | Data roles | Big-picture diagram |
| 2 | Data lake / swamp / lakehouse | Lab 0 recall (bronze is a governed lake) |
| 2 | Medallion bronze / silver / gold | Lab 0-2 (the whole build) |
| 2 | File formats (JSON, Parquet) | bronze JSON -> silver Parquet |
| 2 | Columnar + compression | Lab 1 recall; Lab 3 partition benchmark |
| 2 | Delta Lake / ACID / time travel | Lab 1 recall (folder overwrite vs Delta Lake) |
| 2 | Governance / quarantine / validation | Lab 0 and Lab 1 recalls + the type/date validation + quarantine step |
| 3 | Pipeline anatomy, control vs data flow | Lab 0 recall |
| 3 | Idempotency | Lab 0 ("skip if file exists"); Lab 3 incremental load |
| 3 | Four ingestion patterns + pagination | Lab 0 recall + the fetch steps |
| 3 | Checksums / run metadata | Lab 0 - SHA-256 manifest for every bronze file |
| 3 | Data profiling | Lab 1 - the per-source profile table |
| 3 | Six data-quality questions + quarantine | Lab 1 - profile + quarantine + dedup / date check |
| 3 | Cleaning operations | Lab 1 silver steps |
| 3 | **ETL vs ELT** | Lab 1 - the same transform built both ways |
| 3 | DuckDB as an ELT engine | Lab 1 ELT step (`read_json_auto`, `COPY ... TO`) |
| 3 | Window functions | Lab 4 - `ROW_NUMBER` top-3 job families per discipline |
| 3 | Gold "one table, one question" | Lab 4 - semantic view + charts |
| 4 | The 7 raw-data problems | Lab 1 - the six source shapes (mixed names, nesting, date formats, duplicates, missing levels) are these problems in the wild |
| 4 | Four-step recipe + grain | Lab 2 - grain declared first |
| 4 | Facts vs dimensions, surrogate keys | Lab 2 - fact_posting + the dim_* keys |
| 4 | Additive / semi / non-additive measures | Lab 2 - posting count vs internship share |
| 4 | Star vs snowflake | Lab 2 - flat dimension vs snowflaked region (worked demo) |
| 4 | SCD Type 1 / Type 2 | Lab 2 - dim_company worked example |
| 4 | Conformed dimensions / drill-across | Lab 2 - dim_date shared by two facts |
| 4 | Kimball / Inmon / Data Vault | Lab 2 recall |
| 4 | Semantic layer | Lab 4 - vw_intern_market |
| 5 (new) | Scalability: partitioning, incremental loads, scaling up vs out | Lab 3 |


## Recap - you reused the whole Week 4 toolkit

| Week-4 concept | Where you used it here |
|---|---|
| Medallion bronze -> silver -> gold | Lab 0 -> 1 -> 2, from scratch on live data |
| 4-step design + grain | declared *one row per posting* before building |
| Facts / dimensions / surrogate keys | `fact_posting` + six `dim_*` tables with ID keys |
| Additive / semi / non-additive | posting count / monthly snapshot / internship share |
| Star vs snowflake | `dim_location` (region -> country -> city); flat star vs snowflake demo |
| SCD Type 2 | `dim_company` before-and-after example |
| Shared dimensions + drill-across | `dim_date` and `dim_company` shared with `fact_company_month` |
| Bridge table | `bridge_posting_category` (a posting can have many job families) |
| Semantic layer | `vw_intern_market` |
| **Scalability (new)** | partition pruning, columnar reads, safe repeatable loads |

You started with the live web and finished with a clean, queryable star that answers a real question
about *your* job market - the same path as Week 4, in a brand-new area, and at scale.

## Your turn

1. **Use the bridge properly.** A posting can list several job families. Use the `bridge_posting_category`
   table in a query to recount the top job families. Does the ranking change?
2. **Add a third fact table.** Add a "median days a posting stays open" measure to `fact_company_month`
   (you will need to download on two different days). Which of the three kinds of number is that?
3. **Re-split the data.** Split `postings` into folders by `country` instead of by `month`. Which charts
   above get faster, and which get slower? Explain it using partition pruning.
4. **Go live for real.** Put a few companies you're interested in into the `ATS_COMPANIES` list, re-run,
   and see who is hiring interns.

> **Final challenge.** In two sentences: which one idea from this notebook would you defend hardest to a
> doubtful manager who says *"just query the raw JSON"* - and why?